# RascalJax quickstart: covariance of xi multipoles in a periodic box

This notebook builds a uniform random catalog in a periodic box with a model correlation function, runs the
RascalJax covariance integrals, post-processes them (with shot-noise rescaling fixed to 1, since there are no
jackknives or mocks here) and compares the result with the analytic Gaussian covariance of the multipoles.

Everything RascalJax needs is:

* a `Catalog` of random points (positions, weights, number of data galaxies they represent),
* a `Sample` (a membership-weighted view of a catalog; here the whole catalog),
* for each correlation function, a `CorrelationInput` with the normalized RR pair counts in the covariance
  bins and an `XiTable` with the correlation function used inside the integrals.

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # only matters on a shared GPU
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import spherical_jn
import rascaljax as rj
print(rj.__version__)

## Model power spectrum and correlation function

A smooth toy $P(k)$ with a Gaussian damping, and $\xi(r)$ from its Hankel transform. The table passed to
RascalJax is a 2D $(s, \mu)$ table; here it is isotropic.

In [ ]:
k = np.linspace(1e-4, 2.0, 40001)
P = 2e4 * (k / 0.1) / (1 + (k / 0.1) ** 3.5) * np.exp(-((k * 3.0) ** 2))

def xi_of_r(r):
    r = np.atleast_1d(r)
    return np.trapezoid(k[None, :] ** 2 * P[None, :] * spherical_jn(0, k[None, :] * r[:, None]), k, axis=1) / (2 * np.pi**2)

s_tab = np.arange(1.0, 251.0, 1.0)
mu_tab = np.linspace(0.025, 0.975, 20)
table = rj.XiTable.from_model(s_tab, mu_tab, np.repeat(xi_of_r(s_tab)[:, None], len(mu_tab), axis=1))
plt.semilogx(s_tab, s_tab**2 * table.isotropic(s_tab)); plt.xlabel("s [Mpc/h]"); plt.ylabel("s^2 xi(s)");

## Catalog, binning and pair counts

For uniform randoms in a periodic box the normalized RR counts are analytic (fraction of pairs per bin).
`n_data` is the number of galaxies the randoms represent: it sets the Gaussian shot noise.

In [ ]:
L = 1000.0
N_r, N_g = 200_000, 100_000.0
rng = np.random.default_rng(2)
cat = rj.Catalog(rng.uniform(0, L, size=(N_r, 3)), np.ones(N_r), n_data=N_g, name="box")
sample = rj.Sample(cat, name="all")

s_edges = np.arange(20.0, 101.0, 10.0)
n_mu, max_l = 20, 2
shell = 4 * np.pi / 3 * (s_edges[1:] ** 3 - s_edges[:-1] ** 3) / L**3
rr = np.repeat((shell / n_mu)[:, None], n_mu, axis=1) * (1 - 1 / N_r)
corr = rj.CorrelationInput(s_edges, rr, table)

## Run

`N2, N3, N4` are the numbers of secondary, tertiary and quaternary draws; `n_loops` passes over all primaries are
grouped into `n_loops / loops_per_sample` independent subsamples (used for the precision-matrix bias correction and
convergence checks). All visible GPUs are used.

In [ ]:
raw = rj.run_cov([sample], {(0, 0): corr}, mode="legendre", max_l=max_l, n_mu=n_mu,
                 N2=10, N3=10, N4=10, n_loops=16, loops_per_sample=1,
                 boxsize=L, xi_cut=250.0, refine_iterations=0, seed=3)
res = rj.post_process(raw)
cov = res["full_theory_covariance"]      # ordering (s, ell): index = i_s * n_ell + i_ell
cov_ls = raw.binning.to_pycorr_order(cov)  # ordering (ell, s)

## Analytic Gaussian covariance for comparison

For an isotropic $P(k)$ in a periodic box (Grieb et al. 2016), $C_{\ell\ell}(s_a, s_b) = \frac{2(2\ell+1)}{V}\int \frac{k^2 dk}{2\pi^2}\,[P(k)+1/\bar n]^2\, \bar j_\ell(k s_a)\bar j_\ell(k s_b)$ with bin-averaged Bessel functions. The three terms of RascalJax map onto $P^2$, $2P/\bar n$ and $1/\bar n^2$.

In [ ]:
n_s, n_ell = raw.binning.n_s, raw.binning.n_ell
n_gal, V = N_g / L**3, L**3
s_fine = [np.linspace(s_edges[i], s_edges[i + 1], 41) for i in range(n_s)]
jb = np.array([[np.trapezoid(s**2 * spherical_jn(ell, k[None, :] * s[:, None]).T, s, axis=1) / np.trapezoid(s**2, s)
                for s in s_fine] for ell in raw.binning.ells])  # (n_ell, n_s, n_k)

def cov_term(weight_k):
    out = np.zeros((n_s, n_ell, n_s, n_ell))
    for il, ell in enumerate(raw.binning.ells):
        for a in range(n_s):
            for b in range(n_s):
                out[a, il, b, il] = (2 * ell + 1) * 2 / V * np.trapezoid(k**2 * weight_k / (2 * np.pi**2) * jb[il, a] * jb[il, b], k)
    return out.reshape(n_s * n_ell, n_s * n_ell)

cov_th = cov_term((P + 1 / n_gal) ** 2)
c2, c3, c4 = res["c2_terms"].sum(axis=0), res["c3_terms"].sum(axis=0), res["c4_term"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for il, ell in enumerate(raw.binning.ells):
    ax[0].plot(raw.binning.s_mid, np.diag(cov)[il::n_ell], "o", label=f"RascalJax ell={ell}")
    ax[0].plot(raw.binning.s_mid, np.diag(cov_th)[il::n_ell], "-", label=f"Gaussian ell={ell}")
ax[0].set_yscale("log"); ax[0].set_xlabel("s"); ax[0].set_ylabel("Var[xi_ell(s)]"); ax[0].legend()
ax[1].plot(raw.binning.s_mid, np.diag(c4)[0::n_ell] / np.diag(cov_term(P**2))[0::n_ell], "o", label="C4 / P^2 term")
ax[1].plot(raw.binning.s_mid, np.diag(c3)[0::n_ell] / np.diag(cov_term(2 * P / n_gal))[0::n_ell], "s", label="C3 / 2P/n term")
ax[1].axhline(1, color="k", lw=0.5); ax[1].set_ylim(0.5, 1.5); ax[1].set_xlabel("s"); ax[1].set_ylabel("ratio (ell=0 diagonal)"); ax[1].legend();

The 3-point term matches to about a percent; the 4-point diagonal is the noisiest quantity of the
estimator (and the exact comparison in `scripts/bruteforce_check.py` shows it is unbiased). The post-processing
output also contains the per-subsample covariances, the bias-corrected precision matrix and `N_eff`:

In [ ]:
print({k: (v.shape if hasattr(v, "shape") else v) for k, v in res.items() if k not in ("convergence",)})